In [9]:
# Stage 1: load, clean, merge generation + weather for one plant
# Works as a script or pasted into a Colab cell.
import pandas as pd
from pathlib import Path
 
PLANT = 1
RAW = Path(r"C:\Users\VICTUS\Documents\College\Semester 3\Machine Learning\Solar Power Generation Data")
OUT = Path(r"C:\Users\VICTUS\Documents\College\Semester 3\Machine Learning\Solar Power Generation Data\processed")     
OUT.mkdir(parents=True, exist_ok=True)

In [10]:
gen = pd.read_csv(RAW / f"Plant_{PLANT}_Generation_Data.csv")
wx = pd.read_csv(RAW / f"Plant_{PLANT}_Weather_Sensor_Data.csv")
 
# Plant 1 generation timestamps are day-first (dd-mm-YYYY HH:MM); the rest are ISO
if PLANT == 1:
    gen["DATE_TIME"] = pd.to_datetime(gen["DATE_TIME"], format="%d-%m-%Y %H:%M")
else:
    gen["DATE_TIME"] = pd.to_datetime(gen["DATE_TIME"])
wx["DATE_TIME"] = pd.to_datetime(wx["DATE_TIME"])

In [11]:
g = gen.groupby("DATE_TIME").agg(
    AC_POWER=("AC_POWER", "sum"),
    DC_POWER=("DC_POWER", "sum"),
    N_INV=("SOURCE_KEY", "nunique"),   # how many inverters reported at this time
)
w = wx.set_index("DATE_TIME")[["AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]]

In [18]:
start = min(g.index.min(), w.index.min())
end = max(g.index.max(), w.index.max())
grid = pd.date_range(start, end, freq="15min")
df = g.join(w, how="outer").reindex(grid)
df.index.name = "DATE_TIME"

bad = df["AC_POWER"].isna() | (df["N_INV"] < 22)
day = (df.index.hour >= 6) & (df.index.hour < 18)
print("Bad rows:", bad.sum(), "| in daytime:", (bad & day).sum())

runs = bad.ne(bad.shift()).cumsum()
g = bad.groupby(runs).agg(["first", "size"])
print("Longest bad runs (15-min steps):")
print(g[g["first"]]["size"].sort_values(ascending=False).head(5))

Bad rows: 207 | in daytime: 42
Longest bad runs (15-min steps):
19    35
39    31
17    16
15    14
11    12
Name: size, dtype: int64


In [17]:
print(f"Range: {start} -> {end}")
print(f"Rows on full grid: {len(df)}  (~{len(df)/96:.1f} days)")
print("\nMissing values per column:")
print(df.isna().sum())
print("\nInverters reporting per timestamp (N_INV):")
print(df["N_INV"].value_counts(dropna=False).sort_index())
print(f"\nMax N_INV = {df['N_INV'].max()}; rows with fewer inverters than max: "
      f"{(df['N_INV'] < df['N_INV'].max()).sum()}")

Range: 2020-05-15 00:00:00 -> 2020-06-17 23:45:00
Rows on full grid: 3264  (~34.0 days)

Missing values per column:
AC_POWER               106
DC_POWER               106
N_INV                  106
AMBIENT_TEMPERATURE     82
MODULE_TEMPERATURE      82
IRRADIATION             82
dtype: int64

Inverters reporting per timestamp (N_INV):
N_INV
4.0        2
10.0      32
14.0      23
16.0       1
18.0      14
19.0       1
20.0       1
21.0      27
22.0    3057
NaN      106
Name: count, dtype: int64

Max N_INV = 22.0; rows with fewer inverters than max: 101


In [14]:
day = df["IRRADIATION"] > 0.1
print(f"\nDaytime rows (irradiation > 0.1): {day.sum()}; "
      f"of which AC_POWER == 0: {(df.loc[day, 'AC_POWER'] == 0).sum()}")


Daytime rows (irradiation > 0.1): 1397; of which AC_POWER == 0: 0


In [15]:
df.to_csv(OUT / f"plant{PLANT}_merged.csv")
print(f"\nSaved {OUT / f'plant{PLANT}_merged.csv'}")


Saved C:\Users\VICTUS\Documents\College\Semester 3\Machine Learning\Solar Power Generation Data\processed\plant1_merged.csv
